In [1]:
import warnings
warnings.filterwarnings(action='ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import koreanize_matplotlib
import seaborn as sns
import sklearn
from sklearn import datasets
sklearn.set_config(display='text')
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler, RobustScaler, MinMaxScaler, Normalizer
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, silhouette_score

# 다항분포 나이브 베이즈(Multinomial Naive Baues) 알고리즘

베르누이 나이브 베이즈는 분류 데이터의 특징이 `0` 또는 `1`로 표현되었을 때 데이터의 출현 여부에 따라서 0 또는 1로 구분되는 데이터에 사용하지만 다항분포 나이브 베이즈는 분류 데이터의 특징이 `출현 횟수`로 표현되었을 때 출현 횟수에 따라 값을 다르게하는 데이터에 사용한다.

영화 리뷰와 레이블(positive 또는 negative)을 활용해서 다항분포 나이브 베이즈 분류로 영화 리뷰가 긍정적인지 부정적인지 분류한다.

## 영화 리뷰 분류에 사용할 데이터를 준비한다.

In [2]:
review_list = [
    {'movie_review' : 'this is great great movie. I will watch again', 'type' : 'positive'},
    {'movie_review' : 'I like this movie', 'type' : 'positive'},
    {'movie_review' : 'amazing movie in this year', 'type' : 'positive'},
    {'movie_review' : 'cool my boyfriend also said the movie is cool', 'type' : 'positive'},
    {'movie_review' : 'awesome of the awesome movie ever', 'type' : 'positive'},
    {'movie_review' : 'shame I wasted money and time', 'type' : 'negative'},
    {'movie_review' : 'regret on this move. I will never never what movie from this director', 'type' : 'negative'},
    {'movie_review' : 'I do not like this movie', 'type' : 'negative'},
    {'movie_review' : 'I do not like actors in this movie', 'type' : 'negative'},
    {'movie_review' : 'boring boring sleeping movie', 'type' : 'negative'}
]

test_feedback_list = [
    {'movie_review': 'great great great movie ever', 'type': 'positive'},
    {'movie_review': 'I like this amazing movie', 'type': 'positive'},
    {'movie_review': 'my boyfriend said great movie ever', 'type': 'positive'},
    {'movie_review': 'cool cool cool', 'type': 'positive'},
    {'movie_review': 'awesome boyfriend said cool movie ever', 'type': 'positive'},
    {'movie_review': 'shame shame shame', 'type': 'negative'},
    {'movie_review': 'awesome director shame movie boring movie', 'type': 'negative'},
    {'movie_review': 'do not like this movie', 'type': 'negative'},
    {'movie_review': 'I do not like this boring movie', 'type': 'negative'},
    {'movie_review': 'aweful terrible boring movie', 'type': 'negative'}
]

학습 데이터(review_list)를 데이터프레임으로 만든다.

In [3]:
df = pd.DataFrame(review_list)
df

,movie_review,type
0,this is great great movie. I will watch again,positive
1,I like this movie,positive
2,amazing movie in this year,positive
3,cool my boyfriend also said the movie is cool,positive
4,awesome of the awesome movie ever,positive
5,shame I wasted money and time,negative
6,regret on this move. I will never never what m...,negative
7,I do not like this movie,negative
8,I do not like actors in this movie,negative
9,boring boring sleeping movie,negative


사이킷런의 다항분포 나이브 베이즈 분류기는 숫자 데이터만 다룰 수 있기 때문에 positive를 1로 negative를 0으로 치환한 label이라는 파생 변수를 추가한다.

In [4]:
encode = LabelEncoder()
df['label'] = encode.fit_transform(df.type)
df

,movie_review,type,label
0,this is great great movie. I will watch again,positive,1
1,I like this movie,positive,1
2,amazing movie in this year,positive,1
3,cool my boyfriend also said the movie is cool,positive,1
4,awesome of the awesome movie ever,positive,1
5,shame I wasted money and time,negative,0
6,regret on this move. I will never never what m...,negative,0
7,I do not like this movie,negative,0
8,I do not like actors in this movie,negative,0
9,boring boring sleeping movie,negative,0


LabelEncoder는 문자열을 기계학습에서 처리 가능하도록 숫자로 변환할 때 ABC 또는 가나다 순서로 숫자로 변환한다.  
LabelEncoder가 자동으로 만들어 주는 것을 사용하지 않고 사용자가 지정하는 방식으로 하려면 `map()` 메소드에 `딕셔너리`형태로 아래와 같이 지정하면 된다.  
`df.type.map({'positive': 0, 'negative': 1})`

학습 데이터를 피처와 레이블로 분리한다.

In [6]:
x = df.movie_review
x

0        this is great great movie. I will watch again
1                                    I like this movie
2                           amazing movie in this year
3        cool my boyfriend also said the movie is cool
4                    awesome of the awesome movie ever
5                        shame I wasted money and time
6    regret on this move. I will never never what m...
7                             I do not like this movie
8                   I do not like actors in this movie
9                         boring boring sleeping movie
Name: movie_review, dtype: str

In [7]:
y_train = df.label
y_train

0    1
1    1
2    1
3    1
4    1
5    0
6    0
7    0
8    0
9    0
Name: label, dtype: int64

## 모든 영화 리뷰에 출현한 단어 개수 만큼의 크기를 가지는 고정된 벡터를 만든다. => 인코딩

모든 데이터에 출현한 단어의 개수 만큼의 크기를 가지는 벡터를 만들고 고정된 벡터로 표현하기 위해 CountVectorizer를 import 한다.

In [8]:
from sklearn.feature_extraction.text import CountVectorizer

In [10]:
cv = CountVectorizer()
x_train = cv.fit_transform(x)
# print(x_train)
encoding = x_train.toarray()
print(encoding)

[[0 1 0 0 0 0 0 0 0 0 0 0 0 2 0 1 0 0 0 1 0 0 0 0 0 0 0 0 0 0 1 0 0 1 0 1
  0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 1 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0
  0]
 [0 0 0 1 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0
  1]
 [0 0 1 0 0 0 0 1 2 0 0 0 0 0 0 1 0 0 0 1 1 0 0 0 0 0 1 0 0 1 0 0 0 0 0 0
  0]
 [0 0 0 0 0 2 0 0 0 0 0 1 0 0 0 0 0 0 0 1 0 0 0 1 0 0 0 0 0 1 0 0 0 0 0 0
  0]
 [0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 1 0 0 0 1 1 0 0 0
  0]
 [0 0 0 0 0 0 0 0 0 1 0 0 1 0 0 0 0 0 1 1 0 2 0 0 1 1 0 0 0 0 2 0 0 0 1 1
  0]
 [0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 1 0 0 1 0 0 1 0 0 0 0 0 0 0 1 0 0 0 0 0
  0]
 [1 0 0 0 0 0 0 0 0 0 1 0 0 0 1 0 1 0 0 1 0 0 1 0 0 0 0 0 0 0 1 0 0 0 0 0
  0]
 [0 0 0 0 0 0 2 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0
  0]]


위의 넘파이 배열에서 볼 수 있듯이 영화 리뷰에서 총 37개의 단어가 발견되어 각 영화 리뷰가 37개 크기의 벡터로 인코딩된 것을 확인할 수 있다.  
binary=True를 생략해서 다항분포 나이브 베이즈의 입력으로 사용하기 위해 영화 리뷰에 중복되는 단어가 있으면 출현한 횟수로 표현된 것을 알 수 있다.

In [11]:
for s in cv.inverse_transform(x_train):
    print(s)

['this' 'is' 'great' 'movie' 'will' 'watch' 'again']
['this' 'movie' 'like']
['this' 'movie' 'amazing' 'in' 'year']
['is' 'movie' 'cool' 'my' 'boyfriend' 'also' 'said' 'the']
['movie' 'the' 'awesome' 'of' 'ever']
['shame' 'wasted' 'money' 'and' 'time']
['this' 'movie' 'will' 'regret' 'on' 'move' 'never' 'what' 'from'
 'director']
['this' 'movie' 'like' 'do' 'not']
['this' 'movie' 'like' 'in' 'do' 'not' 'actors']
['movie' 'boring' 'sleeping']


In [12]:
print(cv.get_feature_names_out())

['actors' 'again' 'also' 'amazing' 'and' 'awesome' 'boring' 'boyfriend'
 'cool' 'director' 'do' 'ever' 'from' 'great' 'in' 'is' 'like' 'money'
 'move' 'movie' 'my' 'never' 'not' 'of' 'on' 'regret' 'said' 'shame'
 'sleeping' 'the' 'this' 'time' 'wasted' 'watch' 'what' 'will' 'year']


## 다항분포 나이브 베이즈 모델을 만들고 학습시킨다.

사이킷런의 다항분포 나이브 베이즈도 베르누이 나이브 베이즈와 같이 라플라스 스무딩을 지원한다.

베르누이 나이브 베이즈 알고리즘을 사용하기 위해 MultinomialNB를 import 한다.

In [13]:
from sklearn.naive_bayes import MultinomialNB

In [14]:
model = MultinomialNB()
model.fit(x_train, y_train)

MultinomialNB()

## 학습된 다항분포 나이브 베이즈 모델을 테스트 데이터로 예측한다.

테스트 데이터(test_feedback_list)를 데이터프레임으로 만들고 positive를 1로 negative를 0으로 치환한 label이라는 파생 변수를 추가한다.

In [15]:
df = pd.DataFrame(test_feedback_list)
df['label'] = encode.fit_transform(df.type)
df

,movie_review,type,label
0,great great great movie ever,positive,1
1,I like this amazing movie,positive,1
2,my boyfriend said great movie ever,positive,1
3,cool cool cool,positive,1
4,awesome boyfriend said cool movie ever,positive,1
5,shame shame shame,negative,0
6,awesome director shame movie boring movie,negative,0
7,do not like this movie,negative,0
8,I do not like this boring movie,negative,0
9,aweful terrible boring movie,negative,0


테스트 데이터를 피처와 레이블로 분리한다.

In [16]:
x = df.movie_review
x

0                 great great great movie ever
1                    I like this amazing movie
2           my boyfriend said great movie ever
3                               cool cool cool
4       awesome boyfriend said cool movie ever
5                            shame shame shame
6    awesome director shame movie boring movie
7                       do not like this movie
8              I do not like this boring movie
9                 aweful terrible boring movie
Name: movie_review, dtype: str

In [17]:
y_test = df.label
y_test

0    1
1    1
2    1
3    1
4    1
5    0
6    0
7    0
8    0
9    0
Name: label, dtype: int64

In [18]:
x_test = cv.transform(x)
print(x_test.toarray())

[[0 0 0 0 0 0 0 0 0 0 0 1 0 3 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0]
 [0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 1 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0
  0]
 [0 0 0 0 0 0 0 1 0 0 0 1 0 1 0 0 0 0 0 1 1 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0
  0]
 [0 0 0 0 0 0 0 0 3 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0]
 [0 0 0 0 0 1 0 1 1 0 0 1 0 0 0 0 0 0 0 1 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0
  0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 3 0 0 0 0 0 0 0 0
  0]
 [0 0 0 0 0 1 1 0 0 1 0 0 0 0 0 0 0 0 0 2 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0
  0]
 [0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 1 0 0 1 0 0 1 0 0 0 0 0 0 0 1 0 0 0 0 0
  0]
 [0 0 0 0 0 0 1 0 0 0 1 0 0 0 0 0 1 0 0 1 0 0 1 0 0 0 0 0 0 0 1 0 0 0 0 0
  0]
 [0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0]]


In [19]:
predict = model.predict(x_test)
print(predict)

[1 1 1 1 1 0 0 0 0 0]


## 모델을 평가한다.

In [20]:
classification = classification_report(y_test, predict)
print(classification)

              precision    recall  f1-score   support

           0       1.00      1.00      1.00         5
           1       1.00      1.00      1.00         5

    accuracy                           1.00        10
   macro avg       1.00      1.00      1.00        10
weighted avg       1.00      1.00      1.00        10

